In [0]:
!pip install xgboost
dbutils.library.restartPython()

In [0]:
%matplotlib inline
import joblib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn import metrics, feature_selection
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, RobustScaler
from sklearn.compose import ColumnTransformer

In [0]:
from pyspark.sql import *
from pyspark.sql.types import  *
from pyspark.sql.window import Window
import pyspark.sql.functions as F
from pyspark.sql.session import SparkSession

In [0]:
spark= SparkSession.builder.appName("Fraud").getOrCreate()
test_df= spark.read.csv("/Volumes/workspace/default/credit_fraud_project/archive/fraudTest.csv", header=True)
train_df= spark.read.csv("/Volumes/workspace/default/credit_fraud_project/archive/fraudTrain.csv", header=True)
cols_to_drop = [
    "_c0",
    "first",
    "last",
    "street",
    "trans_num",
    "unix_time"
]

train_df = train_df.drop(*cols_to_drop)
test_df = test_df.drop(*cols_to_drop)
max_date= train_df.select(
    F.max(F.to_date('trans_date_trans_time'))
    ).first()[0]

train_df= train_df.withColumns({
    'trans_date_trans_time':F.col("trans_date_trans_time").cast(TimestampType()), 
    'cc_num':F.col("cc_num").cast(LongType()), 
    'amt':F.col("amt").cast(DoubleType()), 
    'zip':F.col("zip").cast(IntegerType()), 
    'lat':F.col("lat").cast(DoubleType()),
    'long':F.col("long").cast(DoubleType()),
    'city_pop':F.col("city_pop").cast(IntegerType()), 
    'dob':F.col("dob").cast(DateType()), 
    'merch_lat':F.col("merch_lat").cast(DoubleType()),
    'merch_long':F.col("merch_long").cast(DoubleType()),
    'is_fraud':F.col("is_fraud").cast(IntegerType()),
    'hour':F.hour('trans_date_trans_time'),
    'month':F.month('trans_date_trans_time'),
    'month':F.weekofyear('trans_date_trans_time')
    })
            
train_df= train_df.withColumn(
    'current_age',
    F.round(
        F.datediff(
            F.lit(max_date)
            , F.col('dob')
        ) / 365, 0
    )
    )
train_df= train_df.drop('dob')

max_date= test_df.select(
    F.max(F.to_date('trans_date_trans_time'))
    ).first()[0]

test_df= test_df.withColumns({
    'trans_date_trans_time':F.col("trans_date_trans_time").cast(TimestampType()), 
    'cc_num':F.col("cc_num").cast(LongType()), 
    'amt':F.col("amt").cast(DoubleType()), 
    'zip':F.col("zip").cast(IntegerType()), 
    'lat':F.col("lat").cast(DoubleType()),
    'long':F.col("long").cast(DoubleType()),
    'city_pop':F.col("city_pop").cast(IntegerType()), 
    'dob':F.col("dob").cast(DateType()), 
    'merch_lat':F.col("merch_lat").cast(DoubleType()),
    'merch_long':F.col("merch_long").cast(DoubleType()),
    'is_fraud':F.col("is_fraud").cast(IntegerType()),
    'hour':F.hour('trans_date_trans_time'),
    'month':F.month('trans_date_trans_time'),
    'month':F.weekofyear('trans_date_trans_time')
    })
            
test_df= test_df.withColumn(
    'current_age',
    F.round(
        F.datediff(
            F.lit(max_date)
            , F.col('dob')
        ) / 365, 0
    )
    )
test_df= test_df.drop('dob')
fraud_trans= train_df.filter(F.col('is_fraud')==1)
non_fraud_trans= train_df.filter(F.col('is_fraud')==0)
fraud_count= fraud_trans.count()
non_fraud_count= non_fraud_trans.count()
fraction= np.divide(fraud_count, non_fraud_count)
sampled_non_fraud= non_fraud_trans.sample(withReplacement=False, fraction=(fraction*1.5), seed=42)
balanced_df= fraud_trans.union(sampled_non_fraud)
balanced_df.groupBy('is_fraud').count()
balanced_df= balanced_df.toPandas()[['amt', 'category', 'hour', 'merchant', 'city', 'current_age', 'job', 'state', 'is_fraud']]
test_df= test_df.toPandas()[['amt', 'category', 'hour', 'merchant', 'city', 'current_age', 'job', 'state', 'is_fraud']]

In [0]:
X= balanced_df.drop(columns=['is_fraud'])
y= balanced_df['is_fraud']

cat_cols= X.select_dtypes(include='object').columns.tolist()
num_cols= X.select_dtypes(exclude='object').columns.tolist()

num_pipeline= Pipeline([
    ('scaler', RobustScaler())
])

# Linear Pipeline

In [0]:
linear_cat_pipeline= Pipeline([
    ('OHE', OneHotEncoder(handle_unknown='ignore', drop='first'))
])


linear_prep= ColumnTransformer(
    [
        ('linear_cat', linear_cat_pipeline, cat_cols),
        ('linear_num', num_pipeline, num_cols)
     ], remainder='passthrough'
)

# Tree Pipeline

In [0]:
tree_cat_pipeline= Pipeline([
    ('Ordinal encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
])

tree_prep= ColumnTransformer(
    [
        ('tree_cat', tree_cat_pipeline, cat_cols),
        ('tree_num', num_pipeline, num_cols)
     ], remainder='passthrough'
)

In [0]:
X_train, X_test, y_train, y_test= train_test_split(X, y, train_size=0.8, stratify=y)
kf= KFold(n_splits=5, shuffle=True, random_state=42)

In [0]:
X_train_linear= linear_prep.fit_transform(X_train)
X_test_linear= linear_prep.transform(X_test)
X_train_tree= tree_prep.fit_transform(X_train)
X_test_tree= tree_prep.transform(X_test)

In [0]:
X_train_linear.shape

In [0]:
estimators= [
    (
        'xgb',GradientBoostingClassifier(learning_rate=np.float64(0.505),loss='exponential',
        n_estimators=300, random_state=42)
    ),
    (
        'gbt', xgb.XGBClassifier(eval_metric='logloss', max_depth=15, n_estimators= 100, objective='binary:logistic', reg_lambda= 5, random_state=42)
    ),
    (
        'et', ExtraTreesClassifier(ccp_alpha=0.0001, max_features=None, n_jobs=-1, random_state=42)
    ),
    (
        'lr', LogisticRegression(max_iter=5000, random_state=42)
    ),
    (
        'rf', RandomForestClassifier(ccp_alpha=0.0001, max_depth=20, n_jobs=-1, random_state=42)
    ),
    (
        'dt', DecisionTreeClassifier(ccp_alpha=0.0001, max_depth=10, random_state= 42)
    )
]

voting= VotingClassifier(estimators,voting='hard',n_jobs=-1)
voting.fit(X_train_tree, y_train)

In [0]:
cm= metrics.confusion_matrix(y_test, voting.predict(X_test_tree))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')

In [0]:
cm= metrics.confusion_matrix(test_df['is_fraud'], voting.predict(tree_prep.transform(test_df.drop(columns=['is_fraud']))))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')

In [0]:
# joblib.load('xgb_tree_prep_estimator').get_params()

In [0]:
pipe= Pipeline([
    ('model',RandomForestClassifier())
])

param_grid= [
    # {
    #     'model':[RandomForestClassifier()],
    #     'model__max_depth':[10,15,20,None],
    #     'model__max_features':['sqrt','log2',None],
    #     'model__ccp_alpha':[1e-4,1e-2,0.2,0.5],
    #     'model__n_jobs':[-1],
    #     'model__random_state':[42]
    # },
    # {
    #     'model':[GradientBoostingClassifier()],
    #     'model__loss':['log_loss','exponential'],
    #     'model__learning_rate':np.linspace(0.01,1,5),
    #     'model__n_estimators':[100,200,300],
    #     'model__random_state':[42]
    # }#,
    {
        'model':[ExtraTreesClassifier()],
        'model__max_depth':[10,15,20,None],
        'model__max_features':['sqrt','log2',None],
        'model__ccp_alpha':[1e-4,1e-2,0.2,0.5],
        'model__n_jobs':[-1],
        'model__random_state':[42]
    }#,
    # {
    #     'model':[DecisionTreeClassifier()],
    #     'model__max_depth':[3,5,10,15,20,None],
    #     'model__max_features':['sqrt','log2',None],
    #     'model__random_state':[42],
    #     'model__ccp_alpha':[1e-4,1e-2,0.2]
    # }
    ]

grid_search= GridSearchCV(pipe, param_grid=param_grid, cv=kf, scoring='f1',n_jobs=-1)
grid_search.fit(X_train_tree, y_train)
print(grid_search.best_params_)
print(grid_search.best_score_)

In [0]:
cm= metrics.confusion_matrix(y_test, grid_search.best_estimator_.predict(X_test_tree))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')

In [0]:
cm= metrics.confusion_matrix(test_df['is_fraud'], grid_search.best_estimator_.predict(tree_prep.transform(test_df.drop(columns=['is_fraud']))))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')